In [0]:
from pyspark.sql.functions import *

silver_path = "/Volumes/workspace/default/mis_datasets_mhealth/silver"

df = spark.read.format("delta").load(silver_path)

display(df)

Métrica de intensidad física -movement_intensity (Usando acelerómetros.)

In [0]:
df_gold = df.withColumn(
    "movement_intensity",
    sqrt(
        pow(col("chest_acc_x"), 2) +
        pow(col("chest_acc_y"), 2) +
        pow(col("chest_acc_z"), 2)
    )
)

In [0]:
display(df_gold)

KPIs por actividad- agregaciones enterprise.

In [0]:
activity_kpis = (
    df_gold.groupBy("activity_name")
    .agg(
        avg("movement_intensity").alias("avg_movement"),
        avg("ecg_signal_avg").alias("avg_ecg"),
        count("*").alias("total_records")
    )
)

In [0]:
display(activity_kpis)

KPIs por usuario

In [0]:
subject_kpis = (
    df_gold.groupBy("subject_id")
    .agg(
        avg("movement_intensity").alias("avg_movement"),
        avg("ecg_signal_avg").alias("avg_ecg"),
        countDistinct("activity_name").alias("activities_performed")
    )
)

In [0]:
display(subject_kpis)

Crear clasificación intensidad

In [0]:
df_gold = df_gold.withColumn(
    "activity_intensity_level",
    when(col("movement_intensity") < 3, "Low")
    .when(col("movement_intensity") < 6, "Medium")
    .otherwise("High")
)

In [0]:
gold_path = "/Volumes/workspace/default/mis_datasets_mhealth/gold"

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .save(gold_path)
)

In [0]:
activity_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/mis_datasets_mhealth/gold_activity_kpis")

In [0]:
subject_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/mis_datasets_mhealth/gold_subject_kpis")